# Part D Evaluation and Interpretation of Findings
ZOU Rongbin (Richard) · CDS3004 Data Mining

This notebook verifies Parts A and B's source data and scores Part C's original outputs.
It reads the standalone survey workbook, `Combined_Survey_and_UCI_Data.xlsx`,
`student_performance_cleaned.csv`, and C's original UCI `split.csv` and `test_predictions.csv`.
Keep these course files together, or set `INPUT_DIR` in the first cell to their location.
Python packages: pandas, NumPy, openpyxl and Matplotlib.

Survey model summary scores come from the saved table in C's notebook. The survey KNN
matrix is transcribed from its labelled figure and is checked against the saved scores.
Every UCI test score is recalculated from C's exported predictions. The notebook's
outputs show aggregate results. Class order follows C's notebook.


## Verify the survey and the separate secondary school EDA

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Set this to the folder with the project inputs if they are stored elsewhere.
INPUT_DIR = Path(os.environ.get('PART_D_INPUT_DIR', '.'))
OUTPUT_DIR = Path(os.environ.get('PART_D_OUTPUT_DIR', '.'))
LABELS = ['below 2.5', '2.5-2.99', '3.0-3.49', '3.5 above']
MODELS = ['Baseline', 'Decision Tree', 'KNN', 'SVM']

survey_path = INPUT_DIR / 'survey_cleaned_python.xlsx'
if not survey_path.is_file():
    survey_path = INPUT_DIR / 'survey_cleaned_python (1).xlsx'
survey = pd.read_excel(survey_path, sheet_name='Cleaned Survey')
workbook = INPUT_DIR / 'Combined_Survey_and_UCI_Data.xlsx'
pd.testing.assert_frame_equal(survey, pd.read_excel(workbook, sheet_name='Survey_Cleaned'))
gpa = survey['the range your Gpa grade of last semester']
hours = survey['How much time you spent on studying per week ( outside of class time)']
counts = gpa.value_counts().reindex(LABELS)
print('A/B survey rows:', len(survey), '| GPA counts:', counts.to_dict())
for name in ('0-5 hours', '16-20 hours'):
    group = gpa[hours.eq(name)]
    print(f'{name}: {(group == LABELS[-1]).sum()}/{len(group)} in the top GPA band')

secondary = pd.read_csv(INPUT_DIR / 'student_performance_cleaned.csv')
print('B secondary-school subject records:', secondary['subject'].value_counts().to_dict())
print('Pooled Pearson r, coded study time and G3:', round(secondary['studytime'].corr(secondary['G3']), 3))
print('Pooled Pearson r, absences and G3:', round(secondary['absences'].corr(secondary['G3']), 3))



A/B survey rows: 46 | GPA counts: {'below 2.5': 3, '2.5-2.99': 5, '3.0-3.49': 21, '3.5 above': 17}
0-5 hours: 5/12 in the top GPA band
16-20 hours: 4/6 in the top GPA band
B secondary-school subject records: {'Portuguese': 649, 'Math': 395}
Pooled Pearson r, coded study time and G3: 0.162
Pooled Pearson r, absences and G3: -0.046


## Recalculate survey KNN metrics from the original confusion matrix

In [2]:
def score_from_confusion(confusion):
    """Rows are actual bands. Columns are predicted bands in LABELS order."""
    array = confusion.to_numpy(dtype=int)
    correct = np.diag(array)
    support = array.sum(axis=1)
    predicted = array.sum(axis=0)
    recall = np.divide(correct, support, out=np.zeros(len(LABELS)), where=support > 0)
    f1 = np.divide(2 * correct, support + predicted, out=np.zeros(len(LABELS)), where=support + predicted > 0)
    return {
        'Correct': int(correct.sum()), 'n': int(array.sum()),
        'Accuracy': float(correct.sum() / array.sum()),
        'Macro F1': float(f1.mean()), 'Balanced accuracy': float(recall.mean()),
    }

# Transcribed from C's saved Survey KNN confusion matrix (comparison cell).
# The saved Survey comparison table supplies other original model scores.
survey_knn = pd.DataFrame([[0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 4, 0], [0, 1, 1, 1]],
                          index=LABELS, columns=LABELS)
survey_summary = pd.DataFrame({'Training CV macro F1': [.154, .290, .313, .249],
                               'Accuracy': [.500, .500, .600, .600],
                               'Macro F1': [.167, .321, .432, .304]}, index=MODELS)
survey_score = score_from_confusion(survey_knn)
assert survey_score['n'] == 10
assert round(survey_score['Macro F1'], 3) == survey_summary.loc['KNN', 'Macro F1']
assert survey_score['Accuracy'] == survey_summary.loc['KNN', 'Accuracy']
print('C survey model comparison (saved results)')
print(survey_summary.to_string())
print('Survey KNN counts and scores recalculated from C’s matrix:', survey_score)



C survey model comparison (saved results)
               Training CV macro F1  Accuracy  Macro F1
Baseline                      0.154       0.5     0.167
Decision Tree                 0.290       0.5     0.321
KNN                           0.313       0.6     0.432
SVM                           0.249       0.6     0.304
Survey KNN counts and scores recalculated from C’s matrix: {'Correct': 6, 'n': 10, 'Accuracy': 0.6, 'Macro F1': 0.4318181818181818, 'Balanced accuracy': 0.5333333333333333}


## Verify the original UCI split and score the exported predictions

In [3]:
uci = pd.read_excel(workbook, sheet_name='UCI_Cleaned')
gpa_codes = {1: LABELS[0], 2: LABELS[0], 3: LABELS[1], 4: LABELS[2], 5: LABELS[3]}
uci_gpa = uci['Last_Semester_GPA'].map(gpa_codes)
split = pd.read_csv(INPUT_DIR / 'split.csv', index_col=0).sort_index()
predictions = pd.read_csv(INPUT_DIR / 'test_predictions.csv', index_col='source_row_zero_based').sort_index()
assert split.index.is_unique and split.index.tolist() == list(uci.index)
assert predictions.index.is_unique and set(split['set']) == {'train', 'test'}
test_rows = split.index[split['set'].eq('test')]
assert test_rows.tolist() == predictions.index.tolist()
pd.testing.assert_series_equal(predictions['actual'], uci_gpa.loc[test_rows], check_names=False)
training_majority = uci_gpa.loc[split.index[split['set'].eq('train')]].mode().item()
assert predictions['Baseline'].eq(training_majority).all()
assert len(test_rows) == 29 and len(uci) - len(test_rows) == 116

uci_confusions = {}
uci_scores = {}
for name in MODELS:
    assert predictions[name].isin(LABELS).all()
    confusion = pd.crosstab(predictions['actual'], predictions[name]).reindex(index=LABELS, columns=LABELS, fill_value=0)
    uci_confusions[name] = confusion
    uci_scores[name] = score_from_confusion(confusion)
uci_summary = pd.DataFrame.from_dict(uci_scores, orient='index')
uci_summary['Training CV macro F1'] = [.138, .190, .267, .209]  # C's saved training CV scores
print('C UCI held-out predictions verified against split and workbook')
print(uci_summary.round(3).to_string())
print('UCI KNN confusion matrix')
print(uci_confusions['KNN'].to_string())



C UCI held-out predictions verified against split and workbook
               Correct   n  Accuracy  Macro F1  Balanced accuracy  Training CV macro F1
Baseline            11  29     0.379     0.138              0.250                 0.138
Decision Tree       12  29     0.414     0.252              0.298                 0.190
KNN                 13  29     0.448     0.280              0.338                 0.267
SVM                 10  29     0.345     0.195              0.244                 0.209
UCI KNN confusion matrix
KNN        below 2.5  2.5-2.99  3.0-3.49  3.5 above
actual                                             
below 2.5          8         1         2          0
2.5-2.99           4         0         1          0
3.0-3.49           3         0         5          0
3.5 above          4         0         1          0


## Plot the original KNN confusion matrices

In [4]:
import matplotlib.pyplot as plt

uci_knn = uci_confusions['KNN']
fig, axes = plt.subplots(1, 2, figsize=(11.4, 5.2))
fig.suptitle('KNN misclassified all test cases in two UCI GPA bands',
             x=.07, y=.98, ha='left', fontsize=17, fontweight='bold', color='#203344')
ticks = ['<2.5', '2.5–2.99', '3.0–3.49', 'Top band']
panels = [(survey_knn, 'Group survey', survey_score), (uci_knn, 'UCI higher education', uci_scores['KNN'])]
for ax, (matrix, name, score) in zip(axes, panels):
    values = matrix.to_numpy()
    ax.imshow(values, cmap='Blues', vmin=0, vmax=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(values[i, j]), ha='center', va='center', fontsize=13,
                    color='white' if values[i, j] >= 5 else '#203344',
                    fontweight='bold' if i == j else 'normal')
    ax.set_xticks(range(4), ticks, fontsize=9)
    ax.set_yticks(range(4), ticks, fontsize=9)
    ax.tick_params(length=0, pad=6)
    ax.set_xlabel('Predicted GPA band', labelpad=8, fontsize=10)
    ax.set_ylabel('Actual GPA band', labelpad=8, fontsize=10)
    ax.set_title(f"{name}   |   {score['Correct']}/{score['n']} correct", loc='left', pad=11,
                 fontsize=12, fontweight='bold', color='#203344')
    for spine in ax.spines.values():
        spine.set_color('#C4D2DF')
fig.text(.07, .045, 'Counts from Part C’s original holdout results. The diagonal shows correct predictions.',
         fontsize=9, color='#425366')
fig.subplots_adjust(left=.08, right=.96, bottom=.19, top=.82, wspace=.36)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / 'D_KNN_Confusion_Matrices.png', dpi=220, facecolor='white')
plt.close(fig)
print('Saved D_KNN_Confusion_Matrices.png')


Saved D_KNN_Confusion_Matrices.png
